## Team6

In [ ]:
# Install libraries
!pip install openai pandas numpy scikit-learn tqdm -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ===========================
# 0. User settings
# ===========================

# ⚠️🚨 [이 부분 api key 또는 경로 변경 필요합니다!!] 🚨⚠️
UPSTAGE_API_KEY = "your-api-key"
EWHA_EMB_PATH      = "./ewha_embeddings.npz"        # 첨부한 이화학칙 임베딩 파일
MMLU_WIKI_EMB_PATH = "./mmlu_wiki_embeddings.npz"   # 첨부한 MMLU 위키 임베딩 파일
TESTSET_PATH       = "./testset.csv"                # 주어진 testset.csv
OUTPUT_PATH        = "./6_final.csv"                # 결과 저장 경로
# ⚠️🚨 [이 부분 api key 또는 경로 변경 필요합니다!!] 🚨⚠️

# ===========================
# (1). Import modules
# ===========================
import os
import re
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from typing import List, Tuple, Dict, Any
from openai import OpenAI
from collections import Counter

# ---------------------------
# (2). Other settings
# ---------------------------
client = OpenAI(api_key=UPSTAGE_API_KEY, base_url="https://api.upstage.ai/v1")
CHAT_MODEL  = "solar-pro2"                 # chat mode
EMBED_MODEL = "embedding-passage"    # use npz


In [ ]:
# ===========================
# 1. Load KB
# ===========================

def load_kb(npz_path: str) -> Tuple[np.ndarray, np.ndarray]:
    """
    Load embedings, chunks from npz file
    """
    data = np.load(npz_path, allow_pickle=True)
    embeddings = data["embeddings"]
    chunks = data["chunks"]
    return embeddings, chunks

# load
ewha_embs, ewha_chunks = load_kb(EWHA_EMB_PATH)
mmlu_embs, mmlu_chunks = load_kb(MMLU_WIKI_EMB_PATH)

print("Ewha KB:", ewha_embs.shape, len(ewha_chunks))
print("MMLU KB:", mmlu_embs.shape, len(mmlu_chunks))


Ewha KB: (56, 4096) 56
MMLU KB: (5208, 4096) 5208


In [ ]:
# ===========================
# 2. Search (Question embedding + cosine similarity)
# ===========================

def embed_query(text: str) -> np.ndarray:
    """
    Upstage Embedding API transforms questions into vectors
    """
    resp = client.embeddings.create(
        model=EMBED_MODEL,
        input=[text]
    )
    emb = np.array(resp.data[0].embedding, dtype="float32")
    return emb

def retrieve_context(
    question: str,
    kb_embeddings: np.ndarray,
    kb_chunks: np.ndarray,
    top_k: int = 5
) -> str:
    """
    Combine the top_k chunk most similar to the question in a given KB
    """
    q_emb = embed_query(question).reshape(1, -1)
    sims = cosine_similarity(q_emb, kb_embeddings)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]
    selected_chunks = [str(kb_chunks[i]) for i in top_idx]
    context = "\n\n---\n\n".join(selected_chunks)
    return context

def retrieve_ewha_context(question: str, top_k: int = 5) -> str:
    return retrieve_context(question, ewha_embs, ewha_chunks, top_k)

def retrieve_mmlu_context(question: str, top_k: int = 5) -> str:
    return retrieve_context(question, mmlu_embs, mmlu_chunks, top_k)


In [ ]:
# ===========================
# 3. Routing (Korean/English discrimination)
# ===========================

def is_korean(text: str) -> bool:
    """Does the string contain Korean characters"""
    return any('\uac00' <= ch <= '\ud7a3' for ch in text)

def route_question_and_get_context(question: str) -> Tuple[str, str]:
    """
    Look at the question:
    - language: 'ko' or 'en'
    - context: Text retrieved from Ewha or MMLU KB
    """
    if is_korean(question):
        language = "ko"
        context = retrieve_ewha_context(question, top_k=5)
    else:
        language = "en"
        context = retrieve_mmlu_context(question, top_k=7)
    return language, context


In [ ]:
from collections import Counter
import re
from typing import List, Dict, Any

# ===========================
# 4. Call Solar-Pro2(chat)
# ===========================

def build_prompt_with_context(question: str, context: str, language: str = "ko") -> List[Dict[str, Any]]:
    """
    Solar-Pro2(chat) prompt
    ---------------------------------------------
    Apply 5-Step Prompt Engineering:
    1. Persona Projection
    2. Task-Specific Prompt
    3. Structured Prompt
    4. Chain of Thought
    5. Data Chunking
    """

    if language == "ko":
        # ======================================================
        # 🇰🇷 Ewha Womans University School Regulations QA Version
        # ======================================================
        system_msg = (
            "너는 이화여자대학교 학칙을 전문적으로 해석하고 문제에 따른 정확한 답을 제시하는 AI 조교야. "
            "법학과 조교 수준의 정확성과 논리성을 지녀야 하며, "
            "질문자가 궁금해하는 규정의 핵심 조항과 적용 근거를 명확히 제시해야 해."
        )

        task_instruction = (
            "너의 임무는 아래 주어진 '학칙 CONTEXT'를 바탕으로 "
            "객관식 문제에 대한 정확한 정답을 찾아내는 것이야. "
            "학칙에 명시되지 않은 정보는 절대로 추론하거나 상상해서는 안 돼."
        )

        structured_input = (
            f"{task_instruction}\n\n"
            "아래 형식에 따라 답변하라:\n"
            "1. CONTEXT 요약 (핵심 조항만)\n"
            "2. QUESTION 분석 (핵심 키워드 중심)\n"
            "3. 근거와 추론 과정 (Chain of Thought)\n"
            "4. 최종 정답: [ANSWER]: (X)\n\n"
            f"=== CONTEXT ===\n{context}\n\n=== QUESTION ===\n{question}\n"
        )

        reasoning_guide = (
            "생각 단계:\n"
            "1단계: CONTEXT에서 핵심 규정 조항을 찾는다.\n"
            "2단계: QUESTION의 핵심 단어와 조항을 연결한다.\n"
            "3단계: 문맥에 따라 정답 선택 근거를 설명한다.\n"
            "4단계: 마지막에 '[ANSWER]: (X)' 형식으로만 출력한다.\n"
        )

        chunking_instruction = (
            "CONTEXT가 길다면 문단 단위로 분할하여(예: 제1조~제5조 / 제6조~제10조) "
            "각 chunk에서 관련 조항을 먼저 찾고, 그중 가장 직접적인 근거만 사용하라."
        )

        user_msg = (
            f"{structured_input}\n"
            f"{reasoning_guide}\n"
            f"{chunking_instruction}"
        )

    else:
        # ======================================================
        # 🇺🇸 MMLU QA version
        # ======================================================
        system_msg = (
            "You are an expert multiple-choice QA assistant specialized in reasoning tasks like MMLU. "
            "You must always reason carefully using the provided CONTEXT only. "
            "Be concise, logical, and choose the correct answer label at the end."
        )

        task_instruction = (
            "Your task is to answer the multiple-choice QUESTION strictly based on the given CONTEXT. "
            "Do not invent facts beyond it. Focus on reasoning steps that justify the correct choice."
        )

        structured_input = (
            f"{task_instruction}\n\n"
            "Follow this structure:\n"
            "1. Summarize relevant facts from CONTEXT\n"
            "2. Analyze the QUESTION keywords\n"
            "3. Step-by-step reasoning to justify the answer\n"
            "4. End strictly with: [ANSWER]: (X)\n\n"
            f"=== CONTEXT ===\n{context}\n\n=== QUESTION ===\n{question}\n"
        )

        reasoning_guide = (
            "Reasoning steps:\n"
            "1. Identify key concept(s) in CONTEXT.\n"
            "2. Match QUESTION keywords to CONTEXT facts.\n"
            "3. Eliminate incorrect choices logically.\n"
            "4. Output only one label in [ANSWER]: (X) format.\n"
        )

        chunking_instruction = (
            "If the CONTEXT is long, process it paragraph by paragraph. "
            "Focus only on the most directly relevant facts when selecting the answer."
        )

        user_msg = (
            f"{structured_input}\n"
            f"{reasoning_guide}\n"
            f"{chunking_instruction}"
        )

    # ----------------------------------------------------------
    # Configuring the Final Message
    # ----------------------------------------------------------
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg},
    ]
    return messages


# ===========================
# 4-2. Solar-Pro2 Call (5 calls + most frequent)
# ===========================

def call_solar_with_context(question: str, context: str, language: str = "ko") -> str:
    """
    Call Solar Pro 2 (chat) — Call 5 times at the same prompt
    [ANSWER]: Returns the mode of (X)
    """
    messages = build_prompt_with_context(question, context, language)
    answers = []

    for _ in range(5):  # 5 calls = Majority vote effect
        resp = client.chat.completions.create(
            model="solar-pro2",
            messages=messages,
            temperature=0,
            max_tokens=1024,
        )
        content = resp.choices[0].message.content
        choice = extract_choice_from_answer(content)
        if choice:
            answers.append(choice)

    if not answers:
        return "(N/A)"

    most_common = Counter(answers).most_common(1)[0][0]
    return most_common


# ===========================
# 5. [ANSWER]: (X) extraction function
# ===========================

def extract_choice_from_answer(text: str) -> str:
    """[ANSWER] from the model's full answer: only (X) is drawn"""
    match = re.search(r'\[ANSWER\]\s*:\s*\(([A-J])\)', text)
    if match:
        return f"({match.group(1)})"
    lines = text.strip().splitlines()
    if lines:
        last = lines[-1]
        match2 = re.search(r'\(([A-J])\)\s*$', last.strip())
        if match2:
            return f"({match2.group(1)})"
    return ""


In [ ]:
# ===========================
# 6. Run testset.csv predictions and create 6_final.csv
# ===========================

def main():
    # 1) testset load
    df = pd.read_csv(TESTSET_PATH)
    print(f"총 {len(df)}개의 문제 로드 완료")

    responses = []   # final predictions

    # 2) KB Routing + Model Call + Majority vote for each question
    for i, row in tqdm(df.iterrows(), total=len(df)):
        question = row["prompts"]  # Name of the column containing the question in testset.csv

        # KB Routing
        language, context = route_question_and_get_context(question)

        # Solar Pro2(chat) 5 calls + one option such as majority → "(A)"
        voted_answer = call_solar_with_context(question, context, language=language)

        # Avoid blank/none
        if not voted_answer:
            voted_answer = "(N/A)"

        responses.append(voted_answer)

    # 3) Configure DataFrame for Final Submission (Question + Predictive Answer)
    df_final = pd.DataFrame({
        "question": df["prompts"],  # a question in testset
        "your_answer": responses       # a predictive answer
    })

    # 4) Save CSV
    df_final.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
    print("최종 파일 생성 완료:", OUTPUT_PATH)


if __name__ == "__main__":
    main()


총 50개의 문제 로드 완료


100%|██████████| 50/50 [17:58<00:00, 21.57s/it]

최종 파일 생성 완료: 6_final.csv
